<a href="https://colab.research.google.com/github/6pb4wnww4g-beep/Jason/blob/main/Jason%20Market%20Projector%20v1.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""
Jason Market Projection v1_1

Formål:
- Leser Solver-ratingene fra Market_Solver_Ratings_v1.
- Leser mu/gamma fra Market_Solver_Summary_v1.
- Leser fremtidige fixtures fra FDR_Overview.
- Beregner projisert Market xG / Opp xG / CS% for GW1-GW5.

Modell:
    Team xG = mu * Attack(team) * DefWeak(opponent) * H/A factor

Der:
    H/A factor = gamma     hvis laget spiller hjemme
               = 1/gamma   hvis laget spiller borte

Viktig:
- Dette er et isolert projeksjonsscript.
- Det endrer ikke TO, Market_xP eller Jason-logikken.
- Dersom FDR_Overview inneholder lag som ikke finnes i Solver-ratingene, legges de i
  Market_Projection_Missing_v1_1 i stedet for at scriptet stopper.

Kjørerekkefølge:
1. Market_Master
2. jason_market_solver_test_v1.py
3. jason_market_projection_v1.py
"""

import math
import re
import time
import numpy as np
import pandas as pd

from google.colab import auth
import gspread
from google.auth import default

# -----------------------
# KONFIGURASJON
# -----------------------

SPREADSHEET_NAME = "Jason Development"

FDR_SHEET = "FDR_Overview"
SOLVER_RATINGS_SHEET = "Market_Solver_Ratings_v1"
SOLVER_SUMMARY_SHEET = "Market_Solver_Summary_v1"

PROJECTION_OUT = "Market_Projection_v1_1"
PROJECTION_WIDE_OUT = "Market_Projection_Wide_v1_1"
MISSING_OUT = "Market_Projection_Missing_v1_1"
SUMMARY_OUT = "Market_Projection_Summary_v1_1"

GW_COLUMNS = ["GW1", "GW2", "GW3", "GW4", "GW5"]

# FPL/FDR-kode -> vanlig markedsnavn.
# Scriptet bruker startswith/fuzzy match mot Solver-ratingene, så navn trenger ikke være 100 % identisk.
CODE_TO_TEAM = {
    "ARS": "Arsenal",
    "AVL": "Aston Villa",
    "BHA": "Brighton and Hove Albion",
    "BOU": "Bournemouth",
    "BRE": "Brentford",
    "BUR": "Burnley",
    "CHE": "Chelsea",
    "CRY": "Crystal Palace",
    "EVE": "Everton",
    "FUL": "Fulham",
    "LEE": "Leeds United",
    "LIV": "Liverpool",
    "MCI": "Manchester City",
    "MUN": "Manchester United",
    "NEW": "Newcastle United",
    "NFO": "Nottingham Forest",
    "SUN": "Sunderland",
    "TOT": "Tottenham Hotspur",
    "WHU": "West Ham United",
    "WOL": "Wolverhampton Wanderers",
    "IPS": "Ipswich Town",
    "COV": "Coventry City",
    "HUL": "Hull City",
}

# -----------------------
# Google Sheets helpers
# -----------------------

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open(SPREADSHEET_NAME)


def parse_float(v, default=np.nan):
    if v is None:
        return default
    s = str(v).strip()
    if s == "":
        return default
    s = s.replace(" ", "")
    if "," in s and "." not in s:
        s = s.replace(",", ".")
    elif "," in s and "." in s:
        s = s.replace(",", "")
    try:
        return float(s)
    except Exception:
        return default


def read_sheet(sheet_name):
    ws = sh.worksheet(sheet_name)
    values = ws.get_all_values()
    if not values:
        return pd.DataFrame()
    header = values[0]
    rows = values[1:]
    return pd.DataFrame(rows, columns=header)


def update_worksheet(sheet_name, dataframe, min_rows=100, min_cols=20, sleep_seconds=1.2):
    print(f"Oppdaterer {sheet_name}...")
    df_clean = dataframe.replace([np.inf, -np.inf], np.nan).fillna("")
    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(title=sheet_name, rows=str(min_rows), cols=str(min_cols))

    ws.resize(
        rows=max(min_rows, len(df_clean) + 20),
        cols=max(min_cols, len(df_clean.columns) + 5),
    )
    ws.update([df_clean.columns.tolist()] + df_clean.values.tolist())
    time.sleep(sleep_seconds)


def norm_name(s):
    s = str(s or "").strip().lower()
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s


# -----------------------
# Les Solver-summary
# -----------------------

summary_df = read_sheet(SOLVER_SUMMARY_SHEET)
if summary_df.empty or not {"Metric", "Value"}.issubset(summary_df.columns):
    raise ValueError(f"{SOLVER_SUMMARY_SHEET} mangler Metric/Value")

summary_map = {str(r["Metric"]).strip(): r["Value"] for _, r in summary_df.iterrows()}
mu = parse_float(summary_map.get("mu / league avg xG"))
gamma = parse_float(summary_map.get("Gamma"))

if pd.isna(mu) or pd.isna(gamma):
    raise ValueError(f"Kunne ikke lese mu/gamma fra {SOLVER_SUMMARY_SHEET}")

print(f"mu: {mu:.4f}")
print(f"gamma: {gamma:.4f}")

# -----------------------
# Les Solver-ratings
# -----------------------

ratings_df = read_sheet(SOLVER_RATINGS_SHEET)
required_rating_cols = ["Team", "Solver Attack", "Solver Def Weakness"]
missing_rating_cols = [c for c in required_rating_cols if c not in ratings_df.columns]
if missing_rating_cols:
    raise ValueError(f"{SOLVER_RATINGS_SHEET} mangler kolonner: {missing_rating_cols}")

ratings_df["Team"] = ratings_df["Team"].astype(str).str.strip()
ratings_df["Solver Attack"] = ratings_df["Solver Attack"].apply(parse_float)
ratings_df["Solver Def Weakness"] = ratings_df["Solver Def Weakness"].apply(parse_float)
ratings_df = ratings_df.dropna(subset=["Solver Attack", "Solver Def Weakness"]).copy()

ratings = {}
name_lookup = {}
for _, r in ratings_df.iterrows():
    team = r["Team"]
    ratings[team] = {
        "attack": float(r["Solver Attack"]),
        "defweak": float(r["Solver Def Weakness"]),
    }
    name_lookup[norm_name(team)] = team


def resolve_team_from_code(code):
    """Returnerer Solver-teamnavn for FDR-kode, eller None hvis rating mangler."""
    code = str(code or "").strip().upper()
    if not code:
        return None

    candidate = CODE_TO_TEAM.get(code)
    if not candidate:
        return None

    n_cand = norm_name(candidate)

    # 1) eksakt normalisert match
    if n_cand in name_lookup:
        return name_lookup[n_cand]

    # 2) startswith begge veier, nyttig for avkuttede navn
    for n, team in name_lookup.items():
        if n.startswith(n_cand) or n_cand.startswith(n):
            return team

    # 3) manuell fallback for vanlige kortnavn
    loose = {
        "brightonandhovealbion": ["brightonandhove", "brighton"],
        "wolverhamptonwanderers": ["wolverhampton", "wolves"],
        "tottenhamhotspur": ["tottenham"],
        "newcastleunited": ["newcastle"],
        "manchesterunited": ["manchesterunited", "manunited"],
        "nottinghamforest": ["nottinghamforest", "nottinghamfore"],
        "westhamunited": ["westham"],
    }
    for key, alts in loose.items():
        if n_cand == key:
            for alt in alts:
                for n, team in name_lookup.items():
                    if n.startswith(alt) or alt.startswith(n):
                        return team

    return None


# -----------------------
# Les FDR_Overview
# -----------------------

fdr_df = read_sheet(FDR_SHEET)
if fdr_df.empty:
    raise ValueError(f"{FDR_SHEET} er tomt")

# Finn Team-kolonnen. Vanligvis heter den Team i kolonne A.
team_col = "Team" if "Team" in fdr_df.columns else fdr_df.columns[0]

missing_gw = [gw for gw in GW_COLUMNS if gw not in fdr_df.columns]
if missing_gw:
    raise ValueError(f"{FDR_SHEET} mangler GW-kolonner: {missing_gw}")


def parse_fdr_cell(value):
    """
    Leser FDR-celle som 'mun 4', 'LEE 2', 'MCI 4'.
    Regel:
      - Opponent med STORE bokstaver = laget har hjemmekamp.
      - Opponent med små bokstaver = laget har bortekamp.
    Returnerer: opponent_code, H/A, fdr_number
    """
    raw = str(value or "").strip()
    if raw == "":
        return None, None, np.nan
    if raw.lower() in {"blank", "bgw", "-", "x"}:
        return None, None, np.nan

    # Fjern parenteser og rare mellomrom, men behold case.
    cleaned = raw.replace("(", " ").replace(")", " ").strip()
    m = re.search(r"([A-Za-z]{2,4})\s*([1-5])?", cleaned)
    if not m:
        return None, None, np.nan

    opp_token = m.group(1)
    fdr_num = parse_float(m.group(2), default=np.nan)

    # Case-regel fra FDR_Overview: uppercase = H, lowercase = A.
    ha = "H" if opp_token.isupper() else "A"
    opp_code = opp_token.upper()
    return opp_code, ha, fdr_num




def poisson_probs(lambda_home, lambda_away, max_goals=12):
    """Returnerer home/draw/away sannsynligheter i prosent fra to xG-lambdaer."""
    ph = [math.exp(-lambda_home) * (lambda_home ** k) / math.factorial(k) for k in range(max_goals + 1)]
    pa = [math.exp(-lambda_away) * (lambda_away ** k) / math.factorial(k) for k in range(max_goals + 1)]
    home = 0.0
    draw = 0.0
    away = 0.0
    for i, p_i in enumerate(ph):
        for j, p_j in enumerate(pa):
            prob = p_i * p_j
            if i > j:
                home += prob
            elif i == j:
                draw += prob
            else:
                away += prob
    total = home + draw + away
    if total > 0:
        home, draw, away = home / total, draw / total, away / total
    return home * 100.0, draw * 100.0, away * 100.0

# -----------------------
# Projection engine
# -----------------------

projection_rows = []
missing_rows = []

for _, row in fdr_df.iterrows():
    team_code = str(row.get(team_col, "")).strip().upper()
    if not team_code or team_code == "TEAM":
        continue

    team_name = resolve_team_from_code(team_code)
    if not team_name:
        missing_rows.append({
            "GW": "ALL",
            "Team Code": team_code,
            "Issue": "Team code finnes ikke i Solver-ratingene",
            "Raw Cell": "",
        })
        continue

    for gw in GW_COLUMNS:
        raw_cell = row.get(gw, "")
        opp_code, ha, fdr_num = parse_fdr_cell(raw_cell)
        if not opp_code:
            continue

        opp_name = resolve_team_from_code(opp_code)
        if not opp_name:
            missing_rows.append({
                "GW": gw,
                "Team Code": team_code,
                "Team": team_name,
                "Opponent Code": opp_code,
                "Issue": "Opponent code finnes ikke i Solver-ratingene",
                "Raw Cell": raw_cell,
            })
            continue

        team_attack = ratings[team_name]["attack"]
        team_defweak = ratings[team_name]["defweak"]
        opp_attack = ratings[opp_name]["attack"]
        opp_defweak = ratings[opp_name]["defweak"]

        if ha == "H":
            hfac_team = gamma
            hfac_opp = 1.0 / gamma
        else:
            hfac_team = 1.0 / gamma
            hfac_opp = gamma

        team_xg = mu * team_attack * opp_defweak * hfac_team
        opp_xg = mu * opp_attack * team_defweak * hfac_opp
        cs_pct = math.exp(-opp_xg) * 100.0
        total_xg = team_xg + opp_xg
        if ha == "H":
            win_pct, draw_pct, loss_pct = poisson_probs(team_xg, opp_xg)
        else:
            # Fra lagets perspektiv: laget er borte, men poisson_probs tar home først.
            opp_home_win, draw_pct, team_away_win = poisson_probs(opp_xg, team_xg)
            win_pct = team_away_win
            loss_pct = opp_home_win

        projection_rows.append({
            "GW": gw,
            "Team Code": team_code,
            "Team": team_name,
            "Opponent Code": opp_code,
            "Opponent": opp_name,
            "H/A": ha,
            "FDR": fdr_num,
            "Proj Team xG": team_xg,
            "Proj Opp xG": opp_xg,
            "Proj CS%": cs_pct,
            "Proj Win%": win_pct,
            "Proj Draw%": draw_pct,
            "Proj Loss%": loss_pct,
            "Proj Total xG": total_xg,
            "Team Attack": team_attack,
            "Team DefWeak": team_defweak,
            "Opp Attack": opp_attack,
            "Opp DefWeak": opp_defweak,
            "mu": mu,
            "gamma": gamma,
            "FDR Raw": raw_cell,
        })

proj_df = pd.DataFrame(projection_rows)
missing_df = pd.DataFrame(missing_rows)

if not proj_df.empty:
    # Sorter naturlig etter GW og lagkode.
    proj_df["GW_num"] = proj_df["GW"].str.extract(r"(\d+)").astype(int)
    proj_df = proj_df.sort_values(["GW_num", "Team Code"]).drop(columns=["GW_num"]).reset_index(drop=True)

    # Wide-format: lett å lese, ett lag per rad.
    wide_rows = []
    for team_code, g in proj_df.groupby("Team Code", sort=True):
        first = g.iloc[0]
        out = {"Team Code": team_code, "Team": first["Team"]}
        for gw in GW_COLUMNS:
            gg = g[g["GW"].eq(gw)]
            if gg.empty:
                out[f"{gw} Opp"] = ""
                out[f"{gw} H/A"] = ""
                out[f"{gw} xG"] = ""
                out[f"{gw} Opp xG"] = ""
                out[f"{gw} CS%"] = ""
                out[f"{gw} Win%"] = ""
                out[f"{gw} Draw%"] = ""
                out[f"{gw} Loss%"] = ""
                out[f"{gw} Total xG"] = ""
            else:
                r = gg.iloc[0]
                out[f"{gw} Opp"] = f'{r["Opponent Code"]} ({r["H/A"]})'
                out[f"{gw} H/A"] = r["H/A"]
                out[f"{gw} xG"] = r["Proj Team xG"]
                out[f"{gw} Opp xG"] = r["Proj Opp xG"]
                out[f"{gw} CS%"] = r["Proj CS%"]
                out[f"{gw} Win%"] = r["Proj Win%"]
                out[f"{gw} Draw%"] = r["Proj Draw%"]
                out[f"{gw} Loss%"] = r["Proj Loss%"]
                out[f"{gw} Total xG"] = r["Proj Total xG"]
        # 5 GW summer / forventninger
        out["Sum xG GW1-5"] = pd.to_numeric(g["Proj Team xG"], errors="coerce").sum()
        out["Avg xG GW1-5"] = pd.to_numeric(g["Proj Team xG"], errors="coerce").mean()
        out["Sum Opp xG GW1-5"] = pd.to_numeric(g["Proj Opp xG"], errors="coerce").sum()
        out["Avg Opp xG GW1-5"] = pd.to_numeric(g["Proj Opp xG"], errors="coerce").mean()
        out["Exp CS GW1-5"] = (pd.to_numeric(g["Proj CS%"], errors="coerce") / 100.0).sum()
        out["Avg CS% GW1-5"] = pd.to_numeric(g["Proj CS%"], errors="coerce").mean()
        out["Avg Win% GW1-5"] = pd.to_numeric(g["Proj Win%"], errors="coerce").mean()
        out["Avg Draw% GW1-5"] = pd.to_numeric(g["Proj Draw%"], errors="coerce").mean()
        out["Avg Loss% GW1-5"] = pd.to_numeric(g["Proj Loss%"], errors="coerce").mean()
        wide_rows.append(out)
    wide_df = pd.DataFrame(wide_rows)
else:
    wide_df = pd.DataFrame()

# Summary
summary_rows = [
    {"Metric": "FDR sheet", "Value": FDR_SHEET},
    {"Metric": "Solver ratings sheet", "Value": SOLVER_RATINGS_SHEET},
    {"Metric": "Solver summary sheet", "Value": SOLVER_SUMMARY_SHEET},
    {"Metric": "mu / league avg xG", "Value": mu},
    {"Metric": "gamma", "Value": gamma},
    {"Metric": "Projection rows", "Value": len(proj_df)},
    {"Metric": "Wide columns", "Value": len(wide_df.columns) if 'wide_df' in globals() and not wide_df.empty else 0},
    {"Metric": "Missing rows", "Value": len(missing_df)},
    {"Metric": "Teams projected", "Value": proj_df["Team Code"].nunique() if not proj_df.empty else 0},
]
summary_out_df = pd.DataFrame(summary_rows)

# Round numeric cols for readability
for df in [proj_df, wide_df, missing_df, summary_out_df]:
    if df is not None and not df.empty:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].round(4)

# -----------------------
# Write sheets
# -----------------------

update_worksheet(SUMMARY_OUT, summary_out_df, min_rows=60, min_cols=10)
update_worksheet(PROJECTION_OUT, proj_df, min_rows=160, min_cols=25)
update_worksheet(PROJECTION_WIDE_OUT, wide_df, min_rows=80, min_cols=40)
update_worksheet(MISSING_OUT, missing_df if not missing_df.empty else pd.DataFrame([{"Status": "Ingen manglende lag/motstandere"}]), min_rows=80, min_cols=15)

print("")
print("FERDIG: Jason Market Projection v1_1_1")
print(f"Projection rows: {len(proj_df)}")
print(f"Missing rows: {len(missing_df)}")
print("")
print("Sjekk arkene:")
print(f"- {SUMMARY_OUT}")
print(f"- {PROJECTION_OUT}")
print(f"- {PROJECTION_WIDE_OUT}")
print(f"- {MISSING_OUT}")

mu: 1.5430
gamma: 1.0865
Oppdaterer Market_Projection_Summary_v1_1...
Oppdaterer Market_Projection_v1_1...
Oppdaterer Market_Projection_Wide_v1_1...
Oppdaterer Market_Projection_Missing_v1_1...

FERDIG: Jason Market Projection v1_1_1
Projection rows: 70
Missing rows: 18

Sjekk arkene:
- Market_Projection_Summary_v1_1
- Market_Projection_v1_1
- Market_Projection_Wide_v1_1
- Market_Projection_Missing_v1_1
